In [1]:
import warnings
warnings.filterwarnings("ignore")

import requests
import pandas as pd
from google.cloud import bigquery

# ── Config ──────────────────────────────────────────────────────────────────
PROJECT_ID   = "orlinappareldataset"
DATASET_ID   = "oms_sale_us"
TABLE_ID     = "sales_orders"

# ── BigQuery Client ──────────────────────────────────────────────────────────
# Uses Application Default Credentials (ADC) — no key file needed.
# If this errors with "Could not automatically determine credentials",
# run this once in a terminal, then restart the kernel:
#     gcloud auth application-default login
client = bigquery.Client(project=PROJECT_ID)

# ── BigQuery Query ───────────────────────────────────────────────────────────
query = f"""
SELECT
    product_sku_code, listing_sku_code, qty, order_status, order_date, total, settlement_amount,
    buyer_city, buyer_state, channel_name, warehouse_name, delivery_date
FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}`
WHERE DATE(order_date) >= '2025-03-18'
AND   DATE(order_date) <= CURRENT_DATE()
"""

print("Fetching BigQuery data...")
df_bq = client.query(query).to_dataframe()
print(f"BQ rows: {len(df_bq)}")

# ── ERP API ──────────────────────────────────────────────────────────────────
BASE_URL = "http://190.92.175.131:8080/DigiBizzErpApi/api/UnknownCallerApi/GetPowerBiReports"
headers = {
    "Report-Api-Token": "aaaqqqwww111",
    "ViewName": "View_Dboard_Design_Master_All_Detail_With_Fabric",
    "CompanyYearId": "83",
    "Accept": "application/json"
}

print("Fetching ERP data...")
response = requests.get(BASE_URL, headers=headers, timeout=60)
response.raise_for_status()
df_erp = pd.DataFrame(response.json())
print(f"ERP rows: {len(df_erp)}")

# ── Merge ────────────────────────────────────────────────────────────────────
# Extract design key: first two segments e.g. "417-03" from "417-03-M"
df_bq["design_key"]  = df_bq["listing_sku_code"].str.strip().str.upper().str.extract(r'^(\d+-\d+)')
df_erp["design_key"] = df_erp["DESIGN_NO"].str.strip().str.upper().str.extract(r'^(\d+-\d+)')

# Keep only needed ERP columns
erp_cols = ["design_key", "DESIGN_NO", "DESIGN_GROUP", "CATALOG_NAME",
            "COLOR", "LAUNCH_DATE", "SECTION", "DESIGNER", "IS_ACTIVE"]
erp_cols = [c for c in erp_cols if c in df_erp.columns]
df_erp_slim = df_erp[erp_cols].drop_duplicates(subset="design_key")

df_merged = df_bq.merge(df_erp_slim, on="design_key", how="left")

df_merged["LAUNCH_DATE"] = pd.to_datetime(df_merged["LAUNCH_DATE"], errors="coerce")
df_merged.drop(columns="design_key", inplace=True)

# ── Summary ──────────────────────────────────────────────────────────────────
print(f"\nMerged rows      : {len(df_merged)}")
print(f"LAUNCH_DATE fill : {df_merged['LAUNCH_DATE'].notna().sum()} / {len(df_merged)}")
print(df_merged[["listing_sku_code", "LAUNCH_DATE"]].head(10))

df_merged.to_csv(r"D:\Demand_Forecasting\merged_sales_erp.csv", index=False)
print("\nSaved to merged_sales_erp.csv")

Fetching BigQuery data...
BQ rows: 2182302
Fetching ERP data...
ERP rows: 9677

Merged rows      : 2182302
LAUNCH_DATE fill : 1572444 / 2182302
  listing_sku_code LAUNCH_DATE
0       444-05-XXL  2025-05-14
1        K-9403-28         NaT
2         444-04-M  2024-12-16
3         444-05-M  2025-05-14
4         498-05-M  2026-03-07
5         549-02-M  2026-03-21
6         474-01-S  2025-04-22
7        474-01-XL  2025-04-22
8         444-03-M  2024-12-16
9        474-01-XL  2025-04-22

Saved to merged_sales_erp.csv


In [1]:
import warnings, pandas as pd
warnings.filterwarnings("ignore")
from google.cloud import bigquery

client = bigquery.Client(project="orlinappareldataset")

print("Fetching sales_orders …")
df = client.query(
    "SELECT * FROM `orlinappareldataset.oms_sale_us.sales_orders`"
).to_dataframe(create_bqstorage_client=True, progress_bar_type=None)

print(f"Rows: {len(df):,} | Columns: {list(df.columns)}")
print(df.head())

Fetching sales_orders …
Rows: 2,503,935 | Columns: ['channel_order_id', 'channel_sub_order_id', 'buyer_name', 'buyer_city', 'buyer_state', 'product_sku_code', 'listing_sku_code', 'channel_name', 'category_name', 'qty', 'total', 'delivery_date', 'order_status', 'return_good_qty', 'return_bad_qty', 'brand_name', 'payment_received', 'listing_fee', 'shipping_fee', 'octroi_fee', 'emi_fee', 'payment_fees', 'commission_fees', 'commission_fee_percent', 'reverse_shipping_fee', 'cancellation_fee', 'fee_discounts', 'pick_and_pack_fee', 'storage_fee', 'settlement_amount', 'service_tax', 'net_profit', 'apology_fee', 'warehouse_name', 'warehouse_alias', 'warehouse_state_name', 'order_date', 'invoice_date', 'promo_discount', 'sub_order_key', 'source', 'shipment_type', 'velocity_created_at', 'cgst_amount', 'sgst_amount', 'igst_amount', 'tcs_amount']
      channel_order_id   channel_sub_order_id buyer_name buyer_city  \
0  `261531371016150784  `261531371016150784_1    Unknown    Unknown   
1  `24919119

In [4]:
df["order_status"].unique().tolist()

['Delivered',
 'New',
 'Rto Delivered',
 'Manifested',
 'Out For Delivery',
 'Reverse Closed',
 'Return Received',
 'Cancelled Return Received',
 'Shipped',
 'Cancel Init',
 'Cancelled Before Shipping',
 'Ready to ship',
 'Cancelled',
 'In Transit',
 'Return Init',
 'Packed',
 'Rto In Transit',
 'Reached At Destination',
 'Reverse Delivered',
 'Undelivered',
 'Reverse In Transit',
 'Partial Return Received',
 'Rto Processing',
 'Partial Cancelled Return Received',
 'Rto Out For Delivery',
 'Return Rejected',
 'Pending',
 'Reverse Out For Delivery',
 'Reverse Out For Pickup',
 'Delayed',
 'Reverse Cancelled',
 'Picked Up',
 'Cancel Request Approved',
 'Reverse Not Picked',
 'Damaged',
 'Rto Undelivered',
 'Reverse Manifest',
 'Processing']

In [12]:
import pandas as pd


# Parameters
start_date = "2026-08-17"
end_date = "2026-08-23"

selected_statuses = ['Delivered',
 'New',
 'Rto Delivered',
 'Manifested',
 'Out For Delivery',
 'Reverse Closed',
 'Return Received',
 'Cancelled Return Received',
 'Shipped',
 'Cancel Init',
 'Ready to ship',
 'In Transit',
 'Return Init',
 'Packed',
 'Rto In Transit',
 'Reached At Destination',
 'Reverse Delivered',
 'Undelivered',
 'Reverse In Transit',
 'Partial Return Received',
 'Rto Processing',
 'Partial Cancelled Return Received',
 'Rto Out For Delivery',
 'Return Rejected',
 'Pending',
 'Reverse Out For Delivery',
 'Reverse Out For Pickup',
 'Delayed',
 'Reverse Cancelled',
 'Picked Up',
 'Cancel Request Approved',
 'Reverse Not Picked',
 'Damaged',
 'Rto Undelivered',
 'Reverse Manifest',
 'Processing']
selected_skus = ['474-01-M']

# Convert date
df["order_date"] = pd.to_datetime(df["order_date"])

# Apply filters
filtered_df = df[
    (df["order_date"] >= start_date) &
    (df["order_date"] <= end_date) &
    (df["order_status"].isin(selected_statuses)) &
    (df["listing_sku_code"].isin(selected_skus))
]

# Overall order count
order_count = filtered_df["channel_order_id"].nunique()

print("Order Count:", order_count)

Order Count: 147


In [ ]:
"""Read-only diagnostic: total sale (units) for one calendar month, computed
the same way the dashboard does — deduplicated `_SOURCE_DF`, Gross Sale
status filter (`data._GROSS_SALE_STATUSES`) — plus a full status breakdown so
you can see exactly what's included/excluded and why the two totals differ.

Run: python check_month_total.py 2026 8
(year and month both required; defaults to the current month if omitted)
"""
from __future__ import annotations

import os
import sys
from datetime import date

# Skip the LGBM background-retrain thread and the daily-refresh timer — this
# is a one-shot read, not a long-running process.
os.environ.setdefault("FORECAST_MODEL", "naive")
os.environ.setdefault("DAILY_REFRESH", "off")

import pandas as pd

import data


def month_total(year: int, month: int) -> None:
    df = data._SOURCE_DF
    if df is None:
        raise SystemExit(
            "data._SOURCE_DF is empty — the one-time rebuild() at import may have "
            "failed; check stderr above for a '[data] real load failed' message."
        )
    # Same de-dup _load_real_plan() applies before any aggregation — the raw
    # source has no order_id, so a small fraction of rows are exact repeats.
    df = df.drop_duplicates()

    start = pd.Timestamp(date(year, month, 1))
    end = start + pd.offsets.MonthEnd(0)
    month_df = df[(df["order_date"] >= start) & (df["order_date"] <= end)]
    status_norm = month_df["order_status"].astype(str).str.strip().str.lower()

    gross_mask = status_norm.isin(data._GROSS_SALE_STATUSES)
    net_mask = status_norm.isin(data._SOLD_STATUSES)

    label = start.strftime("%B %Y")
    print(f"=== {label} ===")
    print(f"All statuses, no exclusion : {month_df['qty'].sum():>10,} units  ({len(month_df):,} rows)")
    print(f"GROSS (dashboard definition): {month_df.loc[gross_mask, 'qty'].sum():>10,} units  ({gross_mask.sum():,} rows)")
    print(f"NET (_SOLD_STATUSES)        : {month_df.loc[net_mask, 'qty'].sum():>10,} units  ({net_mask.sum():,} rows)")

    print()
    print("Status breakdown (qty desc):")
    breakdown = month_df.groupby(status_norm)["qty"].agg(["count", "sum"]).sort_values("sum", ascending=False)
    for status, row in breakdown.iterrows():
        flag = "GROSS" if status in data._GROSS_SALE_STATUSES else "excluded"
        print(f"  {status:<28s} rows={int(row['count']):>7,}  qty={int(row['sum']):>7,}  [{flag}]")


if __name__ == "__main__":
    today = date.today()
    year = int(sys.argv[1]) if len(sys.argv) > 1 else today.year
    month = int(sys.argv[2]) if len(sys.argv) > 2 else today.month
    month_total(year, month)


ModuleNotFoundError: No module named 'data'

In [2]:

df.to_csv("sales_orders.csv", index=False)   

In [9]:
# BigQuery DATE columns come back as the db_dtypes "dbdate" extension type,
# which rejects fillna(0). Convert any date/datetime columns to plain
# datetime64 first, then fill 0 only on the non-date columns.

date_cols = [
    c for c in df_merged.columns
    if str(df_merged[c].dtype).startswith(("datetime", "dbdate", "dbtime"))
]
for c in date_cols:
    df_merged[c] = pd.to_datetime(df_merged[c], errors="coerce")

# Fill 0 on everything except the date columns
non_date_cols = df_merged.columns.difference(date_cols)
df_merged[non_date_cols] = df_merged[non_date_cols].fillna(0)

df_merged.isnull().sum()

product_sku_code          0
listing_sku_code          0
qty                       0
order_status              0
order_date                0
total                     0
settlement_amount         0
buyer_city                0
buyer_state               0
channel_name              0
warehouse_name            0
delivery_date             0
DESIGN_NO                 0
DESIGN_GROUP              0
COLOR                     0
LAUNCH_DATE          607238
SECTION                   0
IS_ACTIVE                 0
dtype: int64

In [10]:
df_merged.fillna(0, inplace=True)
df_merged.isnull().sum()

product_sku_code     0
listing_sku_code     0
qty                  0
order_status         0
order_date           0
total                0
settlement_amount    0
buyer_city           0
buyer_state          0
channel_name         0
warehouse_name       0
delivery_date        0
DESIGN_NO            0
DESIGN_GROUP         0
COLOR                0
LAUNCH_DATE          0
SECTION              0
IS_ACTIVE            0
dtype: int64

In [14]:
df_merged

,product_sku_code,listing_sku_code,qty,order_status,order_date,total,settlement_amount,buyer_city,buyer_state,channel_name,warehouse_name,delivery_date,DESIGN_NO,DESIGN_GROUP,CATALOG_NAME,COLOR,LAUNCH_DATE,SECTION,DESIGNER,IS_ACTIVE
0,444-05-S,444-05-S,1,Cancelled,2025-09-01,1367.05,-266.16,BENGALURU,KARNATAKA,DiEGO International - Amazon DIEGO,DEFAULT 34455,0,444-05,ANARKALI KPD,444,Coffee,2025-05-14 00:00:00,Top,0,True
1,444-08-L,444-08-L,1,Return Received,2026-03-27,1798.00,-248.35,BENGALURU,KARNATAKA,DiEGO International - Amazon DIEGO,DEFAULT 34455,"March 31, 2026 11:56:27 AM",444-08,ANARKALI KPD,444,Berry,2025-09-22 00:00:00,Top,0,True
2,444-07-S,444-07-S,1,Return Received,2026-03-29,1799.00,-248.34,Kathipudi,ANDHRA PRADESH,DiEGO International - Amazon DIEGO,DEFAULT 34455,"April 07, 2026 11:06:51 PM",444-07,ANARKALI KPD,444,Rust,2025-09-22 00:00:00,Top,0,True
3,444-07-XXL,444-07-XXL,1,Return Received,2026-03-11,1799.00,-247.17,PUNE,MAHARASHTRA,DiEGO International - Amazon DIEGO,DEFAULT 34455,"March 13, 2026 01:59:48 PM",444-07,ANARKALI KPD,444,Rust,2025-09-22 00:00:00,Top,0,True
4,444-08-L,444-08-L,1,Return Received,2026-02-05,1798.00,-247.16,GURUGRAM,HARYANA,DiEGO International - Amazon DIEGO,DEFAULT 34455,"February 09, 2026 08:06:31 PM",444-08,ANARKALI KPD,444,Berry,2025-09-22 00:00:00,Top,0,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2018724,027-02-XL,027-02-XL,1,Delivered,2026-03-17,1997.00,1399.19,Memari,WB,Turritopsis - Myntra PPMP,DEFAULT 34455,"March 23, 2026 10:56:04 AM",027-02,ANARKALI KPD,027,Mustard,2026-02-21 00:00:00,Top,0,True
2018725,006-01-XXL,006-01-XXL,1,Delivered,2025-11-13,2228.00,1428.74,Jammu,JK,Turritopsis - Myntra PPMP,DEFAULT 34455,"November 19, 2025 10:57:24 AM",006-01,ANARKALI KPD,006,ORENGE,2025-11-05 00:00:00,Top,0,True
2018726,018-02-XXL,018-02-XXL,1,Delivered,2026-03-01,2079.00,1584.25,New Delhi,DL,Turritopsis - Myntra PPMP,DEFAULT 34455,"March 03, 2026 06:51:54 PM",018-02,ANARKALI KPD,018,Tropical Blue,2026-02-14 00:00:00,Top,0,True
2018727,018-04-L,018-04-L-A,1,Delivered,2026-02-25,2079.00,1588.97,Bangalore,KA,Turritopsis - Myntra PPMP,DEFAULT 34455,"February 28, 2026 04:31:04 PM",018-04,ANARKALI KPD,018,Chocolate Brown,2026-02-14 00:00:00,Top,0,True


In [5]:
check_cols = ["LAUNCH_DATE", "DESIGN_NO", "DESIGN_GROUP", "CATALOG_NAME", "COLOR"]

null_df = df_merged[df_merged[check_cols].isnull().any(axis=1)][["listing_sku_code", "DESIGN_NO"] + check_cols]
print(f"Rows with nulls in checked columns: {len(null_df)}\n")
print(null_df.to_string(index=False))                     

KeyError: "['CATALOG_NAME'] not in index"

In [3]:
df_merged = df_merged.drop(["DESIGNER", "IS_ACTIVE"], axis=1)

KeyError: "['DESIGNER'] not found in axis"

In [17]:
df_merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2018729 entries, 0 to 2018728
Data columns (total 18 columns):
 #   Column             Dtype         
---  ------             -----         
 0   product_sku_code   object        
 1   listing_sku_code   object        
 2   qty                Int64         
 3   order_status       object        
 4   order_date         datetime64[ns]
 5   total              float64       
 6   settlement_amount  float64       
 7   buyer_city         object        
 8   buyer_state        object        
 9   channel_name       object        
 10  warehouse_name     object        
 11  delivery_date      object        
 12  DESIGN_NO          object        
 13  DESIGN_GROUP       object        
 14  CATALOG_NAME       object        
 15  COLOR              object        
 16  LAUNCH_DATE        object        
 17  SECTION            object        
dtypes: Int64(1), datetime64[ns](1), float64(2), object(14)
memory usage: 279.2+ MB


In [18]:
import pandas as pd

df_merged 

,product_sku_code,listing_sku_code,qty,order_status,order_date,total,settlement_amount,buyer_city,buyer_state,channel_name,warehouse_name,delivery_date,DESIGN_NO,DESIGN_GROUP,CATALOG_NAME,COLOR,LAUNCH_DATE,SECTION
0,444-05-S,444-05-S,1,Cancelled,2025-09-01,1367.05,-266.16,BENGALURU,KARNATAKA,DiEGO International - Amazon DIEGO,DEFAULT 34455,0,444-05,ANARKALI KPD,444,Coffee,2025-05-14 00:00:00,Top
1,444-08-L,444-08-L,1,Return Received,2026-03-27,1798.00,-248.35,BENGALURU,KARNATAKA,DiEGO International - Amazon DIEGO,DEFAULT 34455,"March 31, 2026 11:56:27 AM",444-08,ANARKALI KPD,444,Berry,2025-09-22 00:00:00,Top
2,444-07-S,444-07-S,1,Return Received,2026-03-29,1799.00,-248.34,Kathipudi,ANDHRA PRADESH,DiEGO International - Amazon DIEGO,DEFAULT 34455,"April 07, 2026 11:06:51 PM",444-07,ANARKALI KPD,444,Rust,2025-09-22 00:00:00,Top
3,444-07-XXL,444-07-XXL,1,Return Received,2026-03-11,1799.00,-247.17,PUNE,MAHARASHTRA,DiEGO International - Amazon DIEGO,DEFAULT 34455,"March 13, 2026 01:59:48 PM",444-07,ANARKALI KPD,444,Rust,2025-09-22 00:00:00,Top
4,444-08-L,444-08-L,1,Return Received,2026-02-05,1798.00,-247.16,GURUGRAM,HARYANA,DiEGO International - Amazon DIEGO,DEFAULT 34455,"February 09, 2026 08:06:31 PM",444-08,ANARKALI KPD,444,Berry,2025-09-22 00:00:00,Top
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2018724,027-02-XL,027-02-XL,1,Delivered,2026-03-17,1997.00,1399.19,Memari,WB,Turritopsis - Myntra PPMP,DEFAULT 34455,"March 23, 2026 10:56:04 AM",027-02,ANARKALI KPD,027,Mustard,2026-02-21 00:00:00,Top
2018725,006-01-XXL,006-01-XXL,1,Delivered,2025-11-13,2228.00,1428.74,Jammu,JK,Turritopsis - Myntra PPMP,DEFAULT 34455,"November 19, 2025 10:57:24 AM",006-01,ANARKALI KPD,006,ORENGE,2025-11-05 00:00:00,Top
2018726,018-02-XXL,018-02-XXL,1,Delivered,2026-03-01,2079.00,1584.25,New Delhi,DL,Turritopsis - Myntra PPMP,DEFAULT 34455,"March 03, 2026 06:51:54 PM",018-02,ANARKALI KPD,018,Tropical Blue,2026-02-14 00:00:00,Top
2018727,018-04-L,018-04-L-A,1,Delivered,2026-02-25,2079.00,1588.97,Bangalore,KA,Turritopsis - Myntra PPMP,DEFAULT 34455,"February 28, 2026 04:31:04 PM",018-04,ANARKALI KPD,018,Chocolate Brown,2026-02-14 00:00:00,Top


In [19]:
df_merged.to_csv("merged_sales_erp.csv", index=False)
print ("Saved to merged_sales_erp.csv")

Saved to merged_sales_erp.csv


In [20]:
from __future__ import annotations
import pandas as pd
import requests
from IPython.display import display

# ── ERP API config ──────────────────────────────────────────────────────────
ERP_URL = "http://190.92.175.131:8080/DigiBizzErpApi/api/UnknownCallerApi/GetPowerBiReports"
API_TOKEN = "aaaqqqwww111"
COMPANY_YEAR_ID = "83"
VIEW_NAME = "View_Program_Planning_For_BI"  # ← change as needed


def fetch_view(view_name: str, company_year_id: str = COMPANY_YEAR_ID, timeout: int = 60) -> pd.DataFrame:
    headers = {
        "Report-Api-Token": API_TOKEN,
        "ViewName": view_name,
        "CompanyYearId": str(company_year_id),
        "Accept": "application/json",
    }
    print(f"Fetching '{view_name}' (CompanyYearId={company_year_id})...")
    try:
        r = requests.get(ERP_URL, headers=headers, timeout=timeout)
        r.raise_for_status()
    except (requests.ConnectionError, requests.Timeout, requests.HTTPError) as e:
        print(f"ERP error: {e}")
        return pd.DataFrame()

    try:
        data = r.json()
    except ValueError:
        print(f"Non-JSON response:\n{r.text[:300]}")
        return pd.DataFrame()

    df = pd.DataFrame(data)
    print(f"✓ {len(df)} rows × {df.shape[1]} columns")
    return df


df = fetch_view(VIEW_NAME)

if not df.empty:
    print("Columns:", list(df.columns))
    display(df)

Fetching 'View_Program_Planning_For_BI' (CompanyYearId=83)...
✓ 7968 rows × 10 columns
Columns: ['DESIGN_NAME', 'LOT_NO', 'SIZE', 'SIZE_COUNT', 'PRO_ORD_PCS_WEEK_WISE', 'PRO_ORD_WIP_QTY', 'ALLO_CALC', 'ALLOCATION_QTY', 'IN_HOUSE_QTY_WIP', 'JOB_WORK_QTY_WIP']


,DESIGN_NAME,LOT_NO,SIZE,SIZE_COUNT,PRO_ORD_PCS_WEEK_WISE,PRO_ORD_WIP_QTY,ALLO_CALC,ALLOCATION_QTY,IN_HOUSE_QTY_WIP,JOB_WORK_QTY_WIP
0,417-01,LT-06796,M,3,0.0,0.0,0.000000,0.000000,6.0,0.0
1,417-01,LT-06972,XXL,5,0.0,0.0,0.000000,0.000000,0.0,46.0
2,417-03,LT-00197,M,3,0.0,0.0,217.576142,72.525380,0.0,0.0
3,417-03,LT-00201,M,2,0.0,0.0,265.799492,132.899746,0.0,0.0
4,417-03,LT-00202,M,1,0.0,0.0,256.662436,256.662436,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...
7963,K-119-02,LT-00589,32,4,60.0,60.0,0.000000,0.000000,0.0,0.0
7964,K-118-01,LT-00597,30,4,30.0,30.0,0.000000,0.000000,0.0,0.0
7965,394-01,LT-00594,L,5,50.0,50.0,0.000000,0.000000,0.0,0.0
7966,394-02,LT-00595,S,5,50.0,50.0,0.000000,0.000000,0.0,0.0


In [21]:
df = fetch_view(VIEW_NAME)

if not df.empty:
    wip_cols = [
        'PRO_ORD_PCS_WEEK_WISE',
        'PRO_ORD_WIP_QTY',
        'ALLO_CALC',
        'ALLOCATION_QTY',
        'IN_HOUSE_QTY_WIP',
        'JOB_WORK_QTY_WIP'
    ]
    # Coerce to numeric in case any column has object dtype
    df[wip_cols] = df[wip_cols].apply(pd.to_numeric, errors='coerce').fillna(0)
    df['TOTAL_WIP_QTY'] = df[wip_cols].sum(axis=1)

    print("Columns:", list(df.columns))
    display(df)

Fetching 'View_Program_Planning_For_BI' (CompanyYearId=83)...
✓ 7968 rows × 10 columns
Columns: ['DESIGN_NAME', 'LOT_NO', 'SIZE', 'SIZE_COUNT', 'PRO_ORD_PCS_WEEK_WISE', 'PRO_ORD_WIP_QTY', 'ALLO_CALC', 'ALLOCATION_QTY', 'IN_HOUSE_QTY_WIP', 'JOB_WORK_QTY_WIP', 'TOTAL_WIP_QTY']


,DESIGN_NAME,LOT_NO,SIZE,SIZE_COUNT,PRO_ORD_PCS_WEEK_WISE,PRO_ORD_WIP_QTY,ALLO_CALC,ALLOCATION_QTY,IN_HOUSE_QTY_WIP,JOB_WORK_QTY_WIP,TOTAL_WIP_QTY
0,417-01,LT-06796,M,3,0.0,0.0,0.000000,0.000000,6.0,0.0,6.000000
1,417-01,LT-06972,XXL,5,0.0,0.0,0.000000,0.000000,0.0,46.0,46.000000
2,417-03,LT-00197,M,3,0.0,0.0,217.576142,72.525380,0.0,0.0,290.101522
3,417-03,LT-00201,M,2,0.0,0.0,265.799492,132.899746,0.0,0.0,398.699238
4,417-03,LT-00202,M,1,0.0,0.0,256.662436,256.662436,0.0,0.0,513.324872
...,...,...,...,...,...,...,...,...,...,...,...
7963,394-01,LT-00594,XXL,5,50.0,50.0,0.000000,0.000000,0.0,0.0,100.000000
7964,394-02,LT-00595,M,5,50.0,50.0,0.000000,0.000000,0.0,0.0,100.000000
7965,394-02,LT-00595,XXL,5,50.0,50.0,0.000000,0.000000,0.0,0.0,100.000000
7966,394-03,LT-00596,M,5,50.0,50.0,0.000000,0.000000,0.0,0.0,100.000000


In [ ]:
df = fetch_view(VIEW_NAME)

if not df.empty:
    wip_cols = [
        'PRO_ORD_PCS_WEEK_WISE',
        'PRO_ORD_WIP_QTY',
        'ALLO_CALC',
        'ALLOCATION_QTY',
        'IN_HOUSE_QTY_WIP',
        'JOB_WORK_QTY_WIP'
    ]
    # Coerce to numeric in case any column has object dtype
    df[wip_cols] = df[wip_cols].apply(pd.to_numeric, errors='coerce').fillna(0)
    df['TOTAL_WIP_QTY'] = df[wip_cols].sum(axis=1)

    print("Columns:", list(df.columns))
    display(df)

Fetching 'View_Program_Planning_For_BI' (CompanyYearId=83)...
✓ 7968 rows × 10 columns
Columns: ['DESIGN_NAME', 'LOT_NO', 'SIZE', 'SIZE_COUNT', 'PRO_ORD_PCS_WEEK_WISE', 'PRO_ORD_WIP_QTY', 'ALLO_CALC', 'ALLOCATION_QTY', 'IN_HOUSE_QTY_WIP', 'JOB_WORK_QTY_WIP', 'TOTAL_WIP_QTY']


,DESIGN_NAME,LOT_NO,SIZE,SIZE_COUNT,PRO_ORD_PCS_WEEK_WISE,PRO_ORD_WIP_QTY,ALLO_CALC,ALLOCATION_QTY,IN_HOUSE_QTY_WIP,JOB_WORK_QTY_WIP,TOTAL_WIP_QTY
0,417-01,LT-06796,M,3,0.0,0.0,0.000000,0.000000,6.0,0.0,6.000000
1,417-01,LT-06972,XXL,5,0.0,0.0,0.000000,0.000000,0.0,46.0,46.000000
2,417-03,LT-00197,M,3,0.0,0.0,217.576142,72.525380,0.0,0.0,290.101522
3,417-03,LT-00201,M,2,0.0,0.0,265.799492,132.899746,0.0,0.0,398.699238
4,417-03,LT-00202,M,1,0.0,0.0,256.662436,256.662436,0.0,0.0,513.324872
...,...,...,...,...,...,...,...,...,...,...,...
7963,394-01,LT-00594,XXL,5,50.0,50.0,0.000000,0.000000,0.0,0.0,100.000000
7964,394-02,LT-00595,M,5,50.0,50.0,0.000000,0.000000,0.0,0.0,100.000000
7965,394-02,LT-00595,XXL,5,50.0,50.0,0.000000,0.000000,0.0,0.0,100.000000
7966,394-03,LT-00596,M,5,50.0,50.0,0.000000,0.000000,0.0,0.0,100.000000


In [22]:
df.to_csv('df.csv', index=False)

In [1]:
from __future__ import annotations
import pandas as pd
import requests
from IPython.display import display

# ── ERP API config ──────────────────────────────────────────────────────────
ERP_URL = "http://195.250.31.101/DigiBizzErpApi/api/UnknownCallerApi/GetPowerBiReports"
API_TOKEN = "aaaqqqwww111"
COMPANY_YEAR_ID = "83"
VIEW_NAME = "View_Dboard_Trans_Open_Lot_Production_Data_Rdp"  # ← change as needed


def fetch_view(view_name: str, company_year_id: str = COMPANY_YEAR_ID, timeout: int = 60) -> pd.DataFrame:
    headers = {
        "Report-Api-Token": API_TOKEN,
        "ViewName": view_name,
        "CompanyYearId": str(company_year_id),
        "Accept": "application/json",
    }
    print(f"Fetching '{view_name}' (CompanyYearId={company_year_id})...")
    try:
        r = requests.get(ERP_URL, headers=headers, timeout=timeout)
        r.raise_for_status()
    except (requests.ConnectionError, requests.Timeout, requests.HTTPError) as e:
        print(f"ERP error: {e}")
        return pd.DataFrame()

    try:
        data = r.json()
    except ValueError:
        print(f"Non-JSON response:\n{r.text[:300]}")
        return pd.DataFrame()

    df1 = pd.DataFrame(data)
    print(f"✓ {len(df1)} rows × {df1.shape[1]} columns")
    return df1


df1 = fetch_view(VIEW_NAME)

if not df1.empty:
    print("Columns:", list(df1.columns))
    display(df1)

Fetching 'View_Dboard_Trans_Open_Lot_Production_Data_Rdp' (CompanyYearId=83)...
✓ 19394 rows × 10 columns
Columns: ['PROCESS', 'PARTY_NAME', 'VOUCHER_DATE', 'LOT_NO', 'DESIGN_NAME', 'ARTICLE_NAME', 'SECTION', 'BAL_PIECES', 'BAL_MTR', 'STATUS']


,PROCESS,PARTY_NAME,VOUCHER_DATE,LOT_NO,DESIGN_NAME,ARTICLE_NAME,SECTION,BAL_PIECES,BAL_MTR,STATUS
0,General Store Out,None,2025-11-30T00:00:00,LT-05322,417-04-DUPATTA,None,Dupatta,30.0,NaN,None
1,Purchase GRN,MAHADEV FASHION,2026-02-02T00:00:00,LT-06329,002-02,"VICHITRA DYED - 15 - STEEL GREY - 56""",Top,NaN,31.0,None
2,Fab. Job GRN,FASHIONSEYE INDIA JOB,2025-11-11T00:00:00,LT-05864,None,340-01 - EMB SLEEVE - 877,None,NaN,1183.0,None
3,Fab. Job GRN,FENY CREATION,2026-02-25T00:00:00,LT-06639,None,005-03 - EMB FRONT AND SLEEVE LACE - 1375 (1),None,NaN,17.0,None
4,General Store Out,None,2025-12-07T00:00:00,LT-05447,320-03,None,Top,18.0,NaN,None
...,...,...,...,...,...,...,...,...,...,...
19389,Fab. Job GRN,FASHIONSEYE INDIA JOB,2026-02-26T00:00:00,LT-06571,None,474-01 - EMB DUPATTA BUTTI WITH LACE - 714,None,NaN,0.4,None
19390,General Store Out,None,2025-12-02T00:00:00,LT-05373,417-03-PANT,None,Bottom,5.0,NaN,None
19391,Fab. Job Receive,KRISHNA CREATION,2025-12-26T00:00:00,LT-05945,None,K-7702 - EMB DUPATTA - 1105,None,NaN,367.5,None
19392,General Store Out,None,2025-12-24T00:00:00,LT-05398,444-08-PANT,None,Bottom,3.0,NaN,None


In [4]:
df1.to_csv('Allocation_Data.csv', index=False)

In [24]:
print(df.columns.tolist())

['DESIGN_NAME', 'LOT_NO', 'SIZE', 'SIZE_COUNT', 'PRO_ORD_PCS_WEEK_WISE', 'PRO_ORD_WIP_QTY', 'ALLO_CALC', 'ALLOCATION_QTY', 'IN_HOUSE_QTY_WIP', 'JOB_WORK_QTY_WIP', 'TOTAL_WIP_QTY']


In [25]:
# ── Merge TOTAL_WIP_QTY + PENDING_QTY_PIECES onto df_merged ──

# 1) Extract design key + size from sku_code  e.g. "417-03-XL" → "417-03", "XL"
_sku = df_merged["listing_sku_code"].astype(str).str.strip().str.upper().str.split("-")
df_merged["_design"] = _sku.str[:2].str.join("-")
df_merged["_size"]   = _sku.str[2].fillna("").str.strip()

# 2) WIP side — df uses DESIGN_NO (not DESIGN_NAME) as the design identifier
wip_cols = ['PRO_ORD_PCS_WEEK_WISE', 'PRO_ORD_WIP_QTY', 'ALLO_CALC',
            'ALLOCATION_QTY', 'IN_HOUSE_QTY_WIP', 'JOB_WORK_QTY_WIP']
df[wip_cols] = df[wip_cols].apply(pd.to_numeric, errors='coerce').fillna(0)
df["TOTAL_WIP_QTY"] = df[wip_cols].sum(axis=1)

wip = (df.assign(
            _design=df1["DESIGN_NO"].astype(str).str.strip().str.upper().str.extract(r'^(\d+-\d+)')[0],
            _size=df["SIZE"].astype(str).str.strip().str.upper()
        )
        .groupby(["_design", "_size"], as_index=False)["TOTAL_WIP_QTY"].sum())

# 3) Pending side — df1 already has one row per (DESIGN_NO, SIZE)
df1["PENDING_QTY_PIECES"] = pd.to_numeric(df1["PENDING_QTY_PIECES"], errors="coerce").fillna(0)
pend = (df1.assign(
            _design=df1["DESIGN_NO"].astype(str).str.strip().str.upper().str.extract(r'^(\d+-\d+)')[0],
            _size=df1["SIZE"].astype(str).str.strip().str.upper()
        )
        .groupby(["_design", "_size"], as_index=False)["PENDING_QTY_PIECES"].sum())

# 4) Left-join both — sales row count stays the same
df_final = (df_merged
            .merge(wip,  on=["_design", "_size"], how="left")
            .merge(pend, on=["_design", "_size"], how="left"))

df_final[["TOTAL_WIP_QTY", "PENDING_QTY_PIECES"]] = \
    df_final[["TOTAL_WIP_QTY", "PENDING_QTY_PIECES"]].fillna(0)
df_final.drop(columns=["_design", "_size"], inplace=True)

print(f"Rows: {len(df_final)}  (preserved: {len(df_final) == len(df_merged)})")
print(f"TOTAL_WIP_QTY matched:      {(df_final['TOTAL_WIP_QTY'] > 0).sum():,}")
print(f"PENDING_QTY_PIECES matched: {(df_final['PENDING_QTY_PIECES'] > 0).sum():,}")
df_final.head()

Rows: 2018729  (preserved: True)
TOTAL_WIP_QTY matched:      343,960
PENDING_QTY_PIECES matched: 818,954


,product_sku_code,listing_sku_code,qty,order_status,order_date,total,settlement_amount,buyer_city,buyer_state,channel_name,warehouse_name,delivery_date,DESIGN_NO,DESIGN_GROUP,CATALOG_NAME,COLOR,LAUNCH_DATE,SECTION,TOTAL_WIP_QTY,PENDING_QTY_PIECES
0,444-05-S,444-05-S,1,Cancelled,2025-09-01,1367.05,-266.16,BENGALURU,KARNATAKA,DiEGO International - Amazon DIEGO,DEFAULT 34455,0,444-05,ANARKALI KPD,444,Coffee,2025-05-14 00:00:00,Top,0.0,30.0
1,444-08-L,444-08-L,1,Return Received,2026-03-27,1798.00,-248.35,BENGALURU,KARNATAKA,DiEGO International - Amazon DIEGO,DEFAULT 34455,"March 31, 2026 11:56:27 AM",444-08,ANARKALI KPD,444,Berry,2025-09-22 00:00:00,Top,0.0,256.0
2,444-07-S,444-07-S,1,Return Received,2026-03-29,1799.00,-248.34,Kathipudi,ANDHRA PRADESH,DiEGO International - Amazon DIEGO,DEFAULT 34455,"April 07, 2026 11:06:51 PM",444-07,ANARKALI KPD,444,Rust,2025-09-22 00:00:00,Top,219.0,434.0
3,444-07-XXL,444-07-XXL,1,Return Received,2026-03-11,1799.00,-247.17,PUNE,MAHARASHTRA,DiEGO International - Amazon DIEGO,DEFAULT 34455,"March 13, 2026 01:59:48 PM",444-07,ANARKALI KPD,444,Rust,2025-09-22 00:00:00,Top,0.0,186.0
4,444-08-L,444-08-L,1,Return Received,2026-02-05,1798.00,-247.16,GURUGRAM,HARYANA,DiEGO International - Amazon DIEGO,DEFAULT 34455,"February 09, 2026 08:06:31 PM",444-08,ANARKALI KPD,444,Berry,2025-09-22 00:00:00,Top,0.0,256.0


In [26]:
df_final.to_csv("final_merged_data.csv", index=False)

In [27]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2018729 entries, 0 to 2018728
Data columns (total 20 columns):
 #   Column              Dtype         
---  ------              -----         
 0   product_sku_code    object        
 1   listing_sku_code    object        
 2   qty                 Int64         
 3   order_status        object        
 4   order_date          datetime64[ns]
 5   total               float64       
 6   settlement_amount   float64       
 7   buyer_city          object        
 8   buyer_state         object        
 9   channel_name        object        
 10  warehouse_name      object        
 11  delivery_date       object        
 12  DESIGN_NO           object        
 13  DESIGN_GROUP        object        
 14  CATALOG_NAME        object        
 15  COLOR               object        
 16  LAUNCH_DATE         object        
 17  SECTION             object        
 18  TOTAL_WIP_QTY       float64       
 19  PENDING_QTY_PIECES  float64       
dtypes:

In [28]:
df_final.isnull().sum()

product_sku_code      0
listing_sku_code      0
qty                   0
order_status          0
order_date            0
total                 0
settlement_amount     0
buyer_city            0
buyer_state           0
channel_name          0
warehouse_name        0
delivery_date         0
DESIGN_NO             0
DESIGN_GROUP          0
CATALOG_NAME          0
COLOR                 0
LAUNCH_DATE           0
SECTION               0
TOTAL_WIP_QTY         0
PENDING_QTY_PIECES    0
dtype: int64

In [ ]:
View_Dboard_Trans_Production_Order_Data_For_Test_BI

In [3]:
from __future__ import annotations
import pandas as pd
import requests
from IPython.display import display

# ── ERP API config ──────────────────────────────────────────────────────────
ERP_URL = "http://195.250.31.101/DigiBizzErpApi/api/UnknownCallerApi/GetPowerBiReports"
API_TOKEN = "aaaqqqwww111"
COMPANY_YEAR_ID = "83"
VIEW_NAME = "View_Dboard_Trans_Open_Lot_Production_Data_Rdp"  # ← change as needed



def fetch_view(view_name: str, company_year_id: str = COMPANY_YEAR_ID, timeout: int = 60) -> pd.DataFrame:
    headers = {
        "Report-Api-Token": API_TOKEN,
        "ViewName": view_name,
        "CompanyYearId": str(company_year_id),
        "Accept": "application/json",
    }
    print(f"Fetching '{view_name}' (CompanyYearId={company_year_id})...")
    try:
        r = requests.get(ERP_URL, headers=headers, timeout=timeout)
        r.raise_for_status()
    except (requests.ConnectionError, requests.Timeout, requests.HTTPError) as e:
        print(f"ERP error: {e}")
        return pd.DataFrame()

    try:
        data = r.json()
    except ValueError:
        print(f"Non-JSON response:\n{r.text[:300]}")
        return pd.DataFrame()

    df2 = pd.DataFrame(data)
    print(f"✓ {len(df2)} rows × {df2.shape[1]} columns")
    return df2


df2 = fetch_view(VIEW_NAME)

if not df2.empty:
    print("Columns:", list(df2.columns))
    display(df2)

Fetching 'View_Dboard_Trans_Open_Lot_Production_Data_Rdp' (CompanyYearId=83)...
✓ 19394 rows × 10 columns
Columns: ['PROCESS', 'PARTY_NAME', 'VOUCHER_DATE', 'LOT_NO', 'DESIGN_NAME', 'ARTICLE_NAME', 'SECTION', 'BAL_PIECES', 'BAL_MTR', 'STATUS']


,PROCESS,PARTY_NAME,VOUCHER_DATE,LOT_NO,DESIGN_NAME,ARTICLE_NAME,SECTION,BAL_PIECES,BAL_MTR,STATUS
0,Cutting Issue,None,2025-12-29T00:00:00,LT-06046,342-01,None,Top,10.0,NaN,Completed
1,General Store Out,None,2025-12-05T00:00:00,LT-05384,417-04-DUPATTA,None,Dupatta,4.0,NaN,None
2,Fab. Job Receive,KRISHNA CREATION,2026-01-09T00:00:00,LT-05632,None,417-04 - EMB NECK - 74,None,NaN,360.00,None
3,Cutting Issue,None,2025-11-01T00:00:00,LT-05260,488-03,None,Top,5.0,NaN,Completed
4,Fab. Job Issue (Emb.) new,FASHIONSEYE INDIA JOB,2025-09-30T00:00:00,LT-05071,None,K-7903 - EMB BUTTI - 1338,None,NaN,0.06,Completed
...,...,...,...,...,...,...,...,...,...,...
19389,General Store Out,None,2025-11-14T00:00:00,LT-05305,417-03-DUPATTA,None,Dupatta,60.0,NaN,None
19390,General Store Out,None,2025-12-21T00:00:00,LT-05231,444-05-PANT,None,Bottom,11.0,NaN,None
19391,General Store Out,None,2025-11-14T00:00:00,LT-05375,417-03-PANT,None,Bottom,43.0,NaN,None
19392,Cutting Issue,None,2026-05-13T00:00:00,LT-07711,417-04-DUPATTA,None,Dupatta,13.0,NaN,Completed


In [4]:
df2["PROCESS"].unique()

array(['Cutting Issue', 'General Store Out', 'Fab. Job Receive',
       'Fab. Job Issue (Emb.) new', 'Purchase Order', 'Cut to Pack Issue',
       'Fab. Job GRN', 'Allocation Process', 'General Store',
       'Stitching Issue', 'Final Barcode Generator', 'Purchase GRN',
       'Cut to Stitching Issue', 'Thread Cutting Store',
       'Only Stitching Issue', 'Dispatch To Warehouse', 'FOB Issue',
       'Fab. Job QC ', 'Finish GRN', 'Purchase FCM', 'Job QC Process',
       'Stitching Dispatch', 'Cutting Receive ', 'Cut To Pack Dispatch'],
      dtype=object)

In [5]:
df2.to_csv('Closed_Lot.csv', index=False)

1.View_Dboard_Trans_Production_Order_Data_For_Test_BI

In [54]:
from __future__ import annotations
import pandas as pd
import requests
from IPython.display import display

# ── ERP API config ──────────────────────────────────────────────────────────
ERP_URL = "http://190.92.175.131:8080/DigiBizzErpApi/api/UnknownCallerApi/GetPowerBiReports"
API_TOKEN = "aaaqqqwww111"
COMPANY_YEAR_ID = "83"
VIEW_NAME = "View_Dboard_Trans_JOB_WORK_ISSUE_RECEIVE_For_Test_BI"  # ← change as needed


def fetch_view(view_name: str, company_year_id: str = COMPANY_YEAR_ID, timeout: int = 60) -> pd.DataFrame:
    headers = {
        "Report-Api-Token": API_TOKEN,
        "ViewName": view_name,
        "CompanyYearId": str(company_year_id),
        "Accept": "application/json",
    }
    print(f"Fetching '{view_name}' (CompanyYearId={company_year_id})...")
    try:
        r = requests.get(ERP_URL, headers=headers, timeout=timeout)
        r.raise_for_status()
    except (requests.ConnectionError, requests.Timeout, requests.HTTPError) as e:
        print(f"ERP error: {e}")
        return pd.DataFrame()

    try:
        data = r.json()
    except ValueError:
        print(f"Non-JSON response:\n{r.text[:300]}")
        return pd.DataFrame()

    df3 = pd.DataFrame(data)
    print(f"✓ {len(df3)} rows × {df3.shape[1]} columns")
    return df3


df3 = fetch_view(VIEW_NAME)

if not df3.empty:
    print("Columns:", list(df3.columns))
    display(df3)

Fetching 'View_Dboard_Trans_JOB_WORK_ISSUE_RECEIVE_For_Test_BI' (CompanyYearId=83)...
✓ 79699 rows × 12 columns
Columns: ['PROCESS', 'PARTY_NAME', 'VOUCHER_DATE', 'SECTION', 'DESIGN_NO', 'SIZE', 'LOT_NO', 'ISSUE_QTY', 'ALTER_QTY', 'ISSUE_BAL_QTY', 'RATE', 'AMOUNT']


,PROCESS,PARTY_NAME,VOUCHER_DATE,SECTION,DESIGN_NO,SIZE,LOT_NO,ISSUE_QTY,ALTER_QTY,ISSUE_BAL_QTY,RATE,AMOUNT
0,Cut To Pack Dispatch,,2025-08-21T00:00:00,Top,444-01,L,RR,43.0,0.0,0.0,0.0,0.0
1,Cut To Pack Dispatch,,2025-08-21T00:00:00,Top,474-03,M,LT-04371,116.0,0.0,0.0,0.0,0.0
2,Cut To Pack Dispatch,,2025-08-21T00:00:00,Top,474-03,S,LT-04371,45.0,0.0,0.0,0.0,0.0
3,Cut To Pack Dispatch,,2025-08-21T00:00:00,Top,474-03,XL,LT-04371,48.0,0.0,0.0,0.0,0.0
4,Cut To Pack Dispatch,,2025-08-21T00:00:00,Top,474-03,XXL,LT-04371,57.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
79694,Only Stitching Issue,WONDERFUL CREATION,2025-10-25T00:00:00,Top,493-02,L,LT-04415,26.0,0.0,0.0,NaN,0.0
79695,Only Stitching Issue,WONDERFUL CREATION,2025-10-25T00:00:00,Top,493-03,M,LT-04416,39.0,0.0,0.0,NaN,0.0
79696,Only Stitching Issue,WONDERFUL CREATION,2025-10-25T00:00:00,Top,493-03,XL,LT-04416,29.0,0.0,0.0,NaN,0.0
79697,Only Stitching Issue,WONDERFUL CREATION,2025-10-25T00:00:00,Top,493-03,XS,LT-04416,29.0,0.0,0.0,NaN,0.0


In [55]:
df3["PROCESS"].unique()

array(['Cut To Pack Dispatch', 'Cut to Pack Issue',
       'Cut to Stitching Issue', 'Job QC Process',
       'Job Work Stitching GRN', 'Only Stitching Issue'], dtype=object)

In [56]:
df3.isnull().sum()

PROCESS              0
PARTY_NAME           0
VOUCHER_DATE         0
SECTION              0
DESIGN_NO            0
SIZE                 0
LOT_NO              13
ISSUE_QTY            0
ALTER_QTY            0
ISSUE_BAL_QTY        0
RATE              6793
AMOUNT           14994
dtype: int64

In [57]:
df3.to_csv('job_work_issue_receive_raw11.csv', index=False)

In [1]:
import os
import json
import requests
import pandas as pd
from datetime import datetime, timezone

# ── ERP Configuration ─────────────────────────────────────────────────────────
# On GCP: set ERP_BASE_URL to the proxy URL (e.g. https://your-ngrok-url.ngrok.io/...)
# Locally: falls back to the direct ERP address
BASE_URL        = os.environ.get("ERP_BASE_URL", "http://190.92.175.131:8080/DigiBizzErpApi/api/UnknownCallerApi/GetPowerBiReports")
API_TOKEN       = os.environ.get("API_TOKEN",    "aaaqqqwww111")
COMPANY_YEAR_ID = os.environ.get("COMPANY_YEAR_ID", "83")
PROXY_KEY       = os.environ.get("PROXY_KEY", "")

ISSUE_PROCESSES = [
    "Cut to Pack Issue",
    "Cut to Stitching Issue",
    "Only Stitching Issue",
]
GRN_PROCESSES = [
    "Cut To Pack Dispatch",
    "Job Work Stitching GRN",
]

MERGE_KEYS          = ["SECTION", "DESIGN_NO", "SIZE", "LOT_NO"]
DEBIT_COMPANY_YEARS = ["83"]

CACHE_PATH = os.path.join(os.getcwd(), "ppc-dashboard", "backend", "cache.json")

print("✅ Configuration loaded.")


# ── Step 1 — Fetch ERP Data ───────────────────────────────────────────────────
def fetch_erp_data(view_name: str, company_year_id: str) -> pd.DataFrame:
    headers = {
        "Report-Api-Token": API_TOKEN,
        "ViewName":         view_name,
        "CompanyYearId":    company_year_id,
        "Accept":           "application/json",
    }
    if PROXY_KEY:
        headers["x-proxy-key"] = PROXY_KEY
    try:
        resp = requests.get(BASE_URL, headers=headers, timeout=60)
        resp.raise_for_status()
        data = resp.json()
        if isinstance(data, list):
            return pd.DataFrame(data)
        elif isinstance(data, dict):
            records = (
                data.get("data")
                or data.get("Data")
                or data.get("records")
                or data.get("Records")
                or [data]
            )
            return pd.DataFrame(records)
        else:
            print(f"  ⚠️ Unexpected response type: {type(data)}")
            return pd.DataFrame()
    except requests.exceptions.HTTPError as e:
        print(f"  ✗ HTTP Error: {e}")
    except requests.exceptions.ConnectionError:
        print(f"  ✗ Connection Error: Could not reach the server.")
    except requests.exceptions.Timeout:
        print(f"  ✗ Request timed out.")
    except json.JSONDecodeError:
        print(f"  ✗ Invalid JSON in response.")
    return pd.DataFrame()


print("📡 Fetching Job Work Issue & GRN data...")
df = fetch_erp_data(
    view_name="View_Dboard_Trans_JOB_WORK_ISSUE_RECEIVE_For_Test_BI",
    company_year_id=COMPANY_YEAR_ID,
)

if df.empty:
    raise RuntimeError("❌ No data fetched — check API connection and credentials.")

df["PROCESS"] = df["PROCESS"].astype(str).str.strip()
print(f"✅ Fetched {len(df):,} records  |  {len(df.columns)} columns")


# ── Step 2 — Split into Issue & GRN ──────────────────────────────────────────
df_issue = df[df["PROCESS"].isin(ISSUE_PROCESSES)].copy()
df_grn   = df[df["PROCESS"].isin(GRN_PROCESSES)].copy()

print(f"📤 Issue records : {len(df_issue):,}")
print(f"📥 GRN records   : {len(df_grn):,}")

unclassified = df[~df["PROCESS"].isin(ISSUE_PROCESSES + GRN_PROCESSES)]["PROCESS"].unique()
if len(unclassified):
    print(f"\n⚠️  Unclassified PROCESS values:")
    for p in unclassified:
        print(f"   → '{p}'")


# ── Step 3 — Rename Columns & Parse Dates ────────────────────────────────────
df_issue = df_issue.rename(columns={"VOUCHER_DATE": "Issue_Date", "ISSUE_QTY": "Issue_QTY"})
df_grn   = df_grn.rename(columns={"VOUCHER_DATE": "GRN_DATE",   "ISSUE_QTY": "Receive_QTY"})

df_issue["Issue_Date"] = pd.to_datetime(df_issue["Issue_Date"], errors="coerce")
df_grn["GRN_DATE"]     = pd.to_datetime(df_grn["GRN_DATE"],     errors="coerce")

for col in MERGE_KEYS:
    df_issue[col] = df_issue[col].astype(str).str.strip()
    df_grn[col]   = df_grn[col].astype(str).str.strip()

print("✅ Columns renamed and dates parsed.")


# ── Step 4 — Aggregate by Merge Keys ─────────────────────────────────────────
issue_grp = df_issue.groupby(MERGE_KEYS, as_index=False).agg(
    ISSUE_PROCESS   = ("PROCESS",    "first"),
    PARTY_NAME      = ("PARTY_NAME", "first"),
    ISSUE_DATE      = ("Issue_Date", "min"),
    TOTAL_ISSUE_QTY = ("Issue_QTY",  "sum"),
    ALTER_QTY       = ("ALTER_QTY",  "sum"),
    RATE            = ("RATE",       "mean"),
    TOTAL_AMOUNT    = ("AMOUNT",     "sum"),
)

grn_grp = df_grn.groupby(MERGE_KEYS, as_index=False).agg(
    GRN_PROCESS   = ("PROCESS",     "first"),
    GRN_DATE      = ("GRN_DATE",    "max"),
    TOTAL_GRN_QTY = ("Receive_QTY", "sum"),
)

print(f"✅ Issue groups : {len(issue_grp):,}")
print(f"✅ GRN groups   : {len(grn_grp):,}")


# ── Step 5 — Merge & Compute Balance / Status ─────────────────────────────────
df_merged = pd.merge(issue_grp, grn_grp, on=MERGE_KEYS, how="left")
df_merged["TOTAL_GRN_QTY"] = df_merged["TOTAL_GRN_QTY"].fillna(0)

df_merged["BALANCE_QTY"] = (
    df_merged["TOTAL_ISSUE_QTY"] - df_merged["TOTAL_GRN_QTY"]
).clip(lower=0)

df_merged["STATUS"] = df_merged["BALANCE_QTY"].apply(
    lambda x: "Pending" if x > 0 else "Completed"
)

today = pd.Timestamp.today().normalize()
df_merged["DAYS_TAKEN"] = (
    df_merged["GRN_DATE"].fillna(today) - df_merged["ISSUE_DATE"]
).dt.days.clip(lower=0).fillna(0).astype(int)


def delay_bucket(days: int) -> str:
    if days <= 3:    return "0-3 Days"
    elif days <= 7:  return "4-7 Days"
    elif days <= 15: return "8-15 Days"
    else:            return "15+ Days"


df_merged["DELAY_BUCKET"] = df_merged["DAYS_TAKEN"].apply(delay_bucket)

df_merged = df_merged[[
    "LOT_NO", "DESIGN_NO", "SECTION", "SIZE",
    "PARTY_NAME",
    "ISSUE_PROCESS", "ISSUE_DATE",  "TOTAL_ISSUE_QTY", "ALTER_QTY", "RATE", "TOTAL_AMOUNT",
    "GRN_PROCESS",   "GRN_DATE",    "TOTAL_GRN_QTY",
    "BALANCE_QTY", "STATUS", "DAYS_TAKEN", "DELAY_BUCKET",
]]
df_merged = df_merged.sort_values("ISSUE_DATE").reset_index(drop=True)

print(f"✅ Merged records  : {len(df_merged):,}")
print(f"📋 Pending         : {(df_merged['STATUS'] == 'Pending').sum():,}")
print(f"📋 Completed       : {(df_merged['STATUS'] == 'Completed').sum():,}")


# ── Step 6 — Extract Vendor List ──────────────────────────────────────────────
vendor_list = sorted(
    df_issue["PARTY_NAME"]
    .astype(str).str.strip().str.upper()
    .replace("", pd.NA).dropna().unique().tolist()
)
print(f"✅ Total Vendors : {len(vendor_list)}")


# ── Step 7 — Fetch Debit Note Data ────────────────────────────────────────────
debit_frames = []
for cy in DEBIT_COMPANY_YEARS:
    print(f"📡 Fetching Debit Notes for CompanyYearId: {cy}...")
    frame = fetch_erp_data(
        view_name="View_Dboard_Trans_Debit_Note_Data_For_BI",
        company_year_id=cy,
    )
    if not frame.empty:
        frame["CompanyYearId"] = cy
        debit_frames.append(frame)
        print(f"   ✓ {len(frame):,} records")
    else:
        print(f"   ⚠️ No data returned for CompanyYearId {cy}")

if not debit_frames:
    raise RuntimeError("❌ No Debit Note data fetched — check API connection.")

df_debit_raw = pd.concat(debit_frames, ignore_index=True)
print(f"\n✅ Total Debit Note records : {len(df_debit_raw):,}")


# ── Step 8 — Clean & Filter Debit Notes ──────────────────────────────────────
df_debit = df_debit_raw.copy()

if "PAREY_NAME" in df_debit.columns and "PARTY_NAME" not in df_debit.columns:
    df_debit = df_debit.rename(columns={"PAREY_NAME": "PARTY_NAME"})

df_debit["PARTY_NAME"]  = df_debit["PARTY_NAME"].astype(str).str.strip().str.upper()
df_merged["PARTY_NAME"] = df_merged["PARTY_NAME"].astype(str).str.strip().str.upper()

cols_to_drop = [c for c in ["VOUCHER_NO", "BILL_NO", "DESIGN", "SIZE", "LOT_NO"] if c in df_debit.columns]
df_debit = df_debit.drop(columns=cols_to_drop)

rename_map = {}
if "VOUCHER_DATE" in df_debit.columns: rename_map["VOUCHER_DATE"] = "DEBIT_DATE"
if "QTY"          in df_debit.columns: rename_map["QTY"]          = "Debit_Qty"
if "RATE"         in df_debit.columns: rename_map["RATE"]         = "Debit_Rate_Per_Pcs"
if "NET_AMOUNT"   in df_debit.columns: rename_map["NET_AMOUNT"]   = "Debit_Note_Amount"
df_debit = df_debit.rename(columns=rename_map)

df_debit["DEBIT_DATE"] = pd.to_datetime(df_debit["DEBIT_DATE"], errors="coerce")
df_debit = df_debit[df_debit["PARTY_NAME"].isin(vendor_list)].copy()

all_debit_vendors = (
    df_debit_raw["PAREY_NAME"] if "PAREY_NAME" in df_debit_raw.columns
    else df_debit_raw["PARTY_NAME"]
).astype(str).str.strip().str.upper().unique()

unmatched = set(all_debit_vendors) - set(vendor_list)
if unmatched:
    print(f"⚠️  {len(unmatched)} vendor(s) in Debit Notes NOT found in Issue data.")

print(f"✅ Debit Note records after filtering : {len(df_debit):,}")


# ── Step 9 — Save cache.json for Node.js server ───────────────────────────────
df_merged["GRN_DATE"] = pd.to_datetime(df_merged["GRN_DATE"], errors="coerce")
df_final = df_merged.reset_index(drop=True)

cache = {
    "generated_at":   datetime.now(timezone.utc).isoformat(),
    "job_work_report": json.loads(df_final.to_json(orient="records", date_format="iso")),
    "debit_notes":     json.loads(df_debit.reset_index(drop=True).to_json(orient="records", date_format="iso")),
}

os.makedirs(os.path.dirname(CACHE_PATH), exist_ok=True)
with open(CACHE_PATH, "w", encoding="utf-8") as f:
    json.dump(cache, f)

print(f"\n🎉 Done!")
print(f"   job_work_report : {len(df_final):,} records")
print(f"   debit_notes     : {len(df_debit):,} records")
print(f"   cache saved  → {CACHE_PATH}")
print(f"   generated at    : {cache['generated_at']}")
df.to_csv('job_work_issue_receive_raw.csv', index=False)

✅ Configuration loaded.
📡 Fetching Job Work Issue & GRN data...
✅ Fetched 82,228 records  |  12 columns
📤 Issue records : 19,591
📥 GRN records   : 35,368

⚠️  Unclassified PROCESS values:
   → 'Job QC Process'
✅ Columns renamed and dates parsed.
✅ Issue groups : 19,461
✅ GRN groups   : 12,663
✅ Merged records  : 19,461
📋 Pending         : 9,379
📋 Completed       : 10,082
✅ Total Vendors : 107
📡 Fetching Debit Notes for CompanyYearId: 83...
   ✓ 9,776 records

✅ Total Debit Note records : 9,776
⚠️  67 vendor(s) in Debit Notes NOT found in Issue data.
✅ Debit Note records after filtering : 9,396

🎉 Done!
   job_work_report : 19,461 records
   debit_notes     : 9,396 records
   cache saved  → d:\Demand_Forecasting\ppc-dashboard\backend\cache.json
   generated at    : 2026-07-10T18:14:19.568965+00:00


## Where & Why Is Production Delayed — AI Diagnosis (per lot)

Trains the job-work delay model (`delay_predict_jw`, an XGBoost classifier in `api/`)
on completed lots, then scores every lot that's still open right now with a delay
probability. For each one it also explains **why**, using the same signals the model
saw: how far it has already run past its design's normal lead time, how long it's been
sitting at its current process stage without moving, and how much of it has actually
come back so far.

In [2]:
# ── AI delay model: train on history, score every open lot ─────────────────
import sys, os

_api_dir = os.path.join(os.getcwd(), "api")
if not os.path.isdir(_api_dir):
    _api_dir = r"d:\Demand_Forecasting\api"
if _api_dir not in sys.path:
    sys.path.insert(0, _api_dir)

import delay_predict_jw

# `df` = the raw Job Work Issue/GRN rows fetched a few cells above, before the
# Issue/GRN split — this is the exact shape delay_predict_jw expects.
result = delay_predict_jw.compute(df_raw=df)

if result is None:
    print("Not enough labeled history yet to train the AI delay model "
          "(needs 100+ completed/overdue lots). Falling back to the saved CSV...")
    result = delay_predict_jw.compute(issue_csv="job_work_issue_receive_raw.csv")

if result is None:
    print("Still no result — check that `df` has LOT_NO / PROCESS / VOUCHER_DATE / "
          "SECTION / DESIGN_NO / ISSUE_QTY columns.")
else:
    m = result["metrics"]
    print(f"Model trained on {m['labeledLots']:,} historical lots "
          f"(labeled 'late' if lead time > {result['targetDays']} days)")
    if "auc" in m:
        print(f"Back-tested on unseen lots: AUC {m['auc']}  |  {m['accuracy']:.0%} correct  "
              f"(base delay rate in that test set: {m['testBaseRate']:.0%})")
    print(f"Open lots scored : {m['openLots']:,}  |  High risk : {m['atRiskHigh']:,}")


Model trained on 2,206 historical lots (labeled 'late' if lead time > 60 days)
Back-tested on unseen lots: AUC 0.58  |  61% correct  (base delay rate in that test set: 28%)
Open lots scored : 391  |  High risk : 123


In [3]:
# ── WHERE + WHY, per lot ─────────────────────────────────────────────────
def _why(r: dict) -> str:
    reasons = []
    overrun = r["ageDays"] - r["expectedLeadDays"]
    if r["alreadyLate"]:
        reasons.append(f"already {overrun}d past the {r['expectedLeadDays']}d this design usually takes")
    elif overrun > 0:
        reasons.append(f"running {overrun}d over its design's usual {r['expectedLeadDays']}d lead time")
    if r.get("currentProcessDays", 0) >= 7:
        reasons.append(f"stuck at '{r.get('currentProcess', '?')}' for {r['currentProcessDays']}d with no movement")
    if r.get("progressPct", 0) < 0.5:
        reasons.append(f"only {r['progressPct']:.0%} of pieces received back so far")
    if not reasons:
        reasons.append("within normal range so far, but its design/section has an elevated history of delay")
    return "; ".join(reasons)


lots_df = pd.DataFrame()
if result is not None and result["lots"]:
    rows = []
    for lot_no, r in result["lots"].items():
        rows.append({
            "Lot":        lot_no,
            "Design":     r["design"],
            "Section":    r["section"],
            "Where":      r.get("currentProcess", r["route"]),
            "Days@Stage": r.get("currentProcessDays", 0),
            "Progress":   r.get("progressPct", 0.0),
            "AI Risk":    r["delayProb"],
            "Band":       r["riskBand"],
            "Why":        _why(r),
        })
    lots_df = (pd.DataFrame(rows)
               .sort_values("AI Risk", ascending=False)
               .reset_index(drop=True))

    disp = lots_df.head(15).copy()
    disp["Progress"] = (disp["Progress"] * 100).round(0).astype(int).astype(str) + "%"
    disp["AI Risk"]  = (disp["AI Risk"] * 100).round(0).astype(int).astype(str) + "%"
    print("Top 15 lots the AI flags as highest delay risk:\n")
    display(disp)
else:
    print("No open lots to score.")


Top 15 lots the AI flags as highest delay risk:

         Lot    Design Section                   Where  Days@Stage Progress AI Risk  Band                                                                                                                                                       Why
0   LT-07989  K-100-03  Bottom       Cut to Pack Issue          68       0%    100%  High         already 22d past the 46d this design usually takes; stuck at 'Cut to Pack Issue' for 68d with no movement; only 0% of pieces received back so far
1   JW-00040    334-01     Top  Cut to Stitching Issue         391       0%    100%  High  already 355d past the 36d this design usually takes; stuck at 'Cut to Stitching Issue' for 391d with no movement; only 0% of pieces received back so far
2   JW-00041    334-02     Top  Cut to Stitching Issue         391       0%    100%  High  already 345d past the 46d this design usually takes; stuck at 'Cut to Stitching Issue' for 391d with no movement; only 0% of pie

In [4]:
# ── Look up the AI's diagnosis for one specific lot ─────────────────────────
def explain_lot(lot_no: str) -> None:
    """Print the AI's where & why diagnosis for one specific lot number."""
    if result is None:
        print("AI model has no result to explain from.")
        return
    lot_no = str(lot_no)
    r = result["lots"].get(lot_no)
    if r is None:
        print(f"Lot '{lot_no}' is not currently open/in-progress (or wasn't found).")
        return
    late_tag = "  [ALREADY LATE]" if r["alreadyLate"] else ""
    print(f"LOT {lot_no}  —  Design {r['design']}  ·  Section {r['section']}")
    print(f"AI delay risk : {r['delayProb']:.0%}  [{r['riskBand']} risk]{late_tag}")
    print(f"WHERE         : {r.get('currentProcess', r['route'])}  "
          f"({r.get('currentProcessDays', 0)}d without moving, "
          f"{r.get('progressPct', 0):.0%} of pieces received back)")
    print(f"WHY           : {_why(r)}")


# Example: explain the single riskiest open lot right now
if result is not None and not lots_df.empty:
    explain_lot(lots_df.iloc[0]["Lot"])
    print("\n(Call explain_lot('<LOT_NO>') for any other lot number.)")


LOT LT-07989  —  Design K-100-03  ·  Section Bottom
AI delay risk : 100%  [High risk]  [ALREADY LATE]
WHERE         : Cut to Pack Issue  (68d without moving, 0% of pieces received back)
WHY           : already 22d past the 46d this design usually takes; stuck at 'Cut to Pack Issue' for 68d with no movement; only 0% of pieces received back so far

(Call explain_lot('<LOT_NO>') for any other lot number.)
